# Задание 6: Интерактивная карта с результатами анализа

В **шестом модуле** мы познакомились с библиотекой **Folium** и научились создавать интерактивные веб-карты.

В этом задании вам предстоит собрать на одной интерактивной карте данные, которые вы подготовили на предыдущих этапах проекта.

На карте должны быть отображены:

- граница исследуемой территории;
- землепользование;
- объекты повседневных функций и остановки общественного транспорта;
- зоны пешеходной доступности от них 5, 10 и 15 минут (используйте уже объединённые зоны доступности, подготовленные в задании 4/5);

В рамках задания вы:

- загружаете подготовленные пространственные данные;
- создаёте базовую интерактивную карту;
- добавляете несколько тематических слоёв;
- добавляете элементы управления картой;
- сохраняете карту в формате .html

> Рекомендуем самостоятельно ознакомиться с документацией библиотеки Folium и примерами её использования: [Folium User Guide](https://python-visualization.github.io/folium/latest/user_guide.html)


## Предлагаемый план работы

### Шаг 0. Импорт библиотек

Перед началом работы импортируйте библиотеки, которые вам потребуются


In [181]:
# ваш код
import geopandas as gpd
import folium
import osmnx as ox

### Шаг 1. Загрузка данных

Загрузите данные, подготовленные в предыдущих заданиях.

- границу исследуемой территории;
- объекты повседневных функций из задания 1;
- слой землепользования из задания 2;
- остановки общественного транспорта из задания 3;
- зоны пешеходной доступности из задания 4 или они же с рассчитаной оценкой доступности из задания 5;


In [182]:
# ваш код
area_name = 'Лефортово, Москва'
area_bounds = ox.geocode_to_gdf(area_name)

grocery = gpd.read_file('../module_1/places.gpkg', layer='grocery')
landuse = gpd.read_file('../module_2/landuse.gpkg')
public_transport = gpd.read_file('../module_3/public_transport.gpkg')
accessibility_zones = gpd.read_file('../data/accessibility_zones_with_population.gpkg')


### Шаг 2. Проверка и подготовка данных

Посмотрите структуру каждого слоя. Удалите лишние поля, оставив только те, которые нужны для отображения и подсказок.


In [183]:
grocery.head()

,element,id,name,shop,amenity,osm_id,geometry
0,node,380729543,Пятёрочка,supermarket,None,380729543,POINT (37.71253 55.75345)
1,node,460418836,Полярная звезда,supermarket,None,460418836,POINT (37.7126 55.76148)
2,node,950289226,Ашан,supermarket,None,950289226,POINT (37.70775 55.7477)
3,node,1142017706,Пятёрочка,supermarket,None,1142017706,POINT (37.7181 55.74939)
4,node,1142017730,None,convenience,None,1142017730,POINT (37.72019 55.74621)


In [184]:
# ваш код
grocery = grocery[['name', 'shop', 'geometry']]
grocery.head()

,name,shop,geometry
0,Пятёрочка,supermarket,POINT (37.71253 55.75345)
1,Полярная звезда,supermarket,POINT (37.7126 55.76148)
2,Ашан,supermarket,POINT (37.70775 55.7477)
3,Пятёрочка,supermarket,POINT (37.7181 55.74939)
4,None,convenience,POINT (37.72019 55.74621)


In [185]:
landuse.head()

,landuse,area_km2,geometry
0,industrial,0.010384,"MULTIPOLYGON (((418670.01 6179933.517, 418550...."
1,retail,0.002315,"MULTIPOLYGON (((419422.593 6179411.973, 419440..."
2,industrial,0.126357,"MULTIPOLYGON (((418755.006 6180386.152, 418839..."
3,residential,0.055381,"MULTIPOLYGON (((419426.976 6179582.791, 419460..."
4,industrial,0.021615,"MULTIPOLYGON (((419251.461 6181634.244, 419223..."


In [186]:
public_transport.head()

,element,id,osm_id,railway,name,highway,transport_type,geometry
0,node,250722843,250722843,tram_stop,Метро «Авиамоторная»,None,tram,POINT (419418.358 6179175.239)
1,node,380403021,380403021,subway_entrance,None,None,subway,POINT (419440.819 6179138.777)
2,node,380403022,380403022,subway_entrance,None,None,subway,POINT (419499.492 6179098.161)
3,node,380403023,380403023,subway_entrance,None,None,subway,POINT (419528.495 6179125.143)
4,node,380403024,380403024,subway_entrance,None,None,subway,POINT (419429.853 6179175.171)


In [187]:
public_transport = public_transport[['name', 'transport_type', 'geometry']]
public_transport.head()

,name,transport_type,geometry
0,Метро «Авиамоторная»,tram,POINT (419418.358 6179175.239)
1,None,subway,POINT (419440.819 6179138.777)
2,None,subway,POINT (419499.492 6179098.161)
3,None,subway,POINT (419528.495 6179125.143)
4,None,subway,POINT (419429.853 6179175.171)


In [188]:
accessibility_zones.head()


,time_min,category,population_sum,outside_population,share_total_population,outside_share_total_population,geometry
0,5,grocery,64960.0,38145.0,63.00,37.00,"POLYGON ((37.70141 55.74902, 37.70004 55.74878..."
1,10,grocery,99675.0,3430.0,96.67,3.33,"POLYGON ((37.69207 55.74838, 37.69187 55.74825..."
2,15,grocery,99675.0,3430.0,96.67,3.33,"POLYGON ((37.69207 55.74838, 37.69187 55.74825..."


In [189]:
# 1. Проверяем исходные CRS
print("area_bounds:", area_bounds.crs)
print("grocery:", grocery.crs)
print("landuse:", landuse.crs)
print("public_transport:", public_transport.crs)
print("accessibility_zones:", accessibility_zones.crs)


# 2. Оцениваем подходящую UTM-проекцию по границе района
target_crs = "EPSG:4326"

print("target_crs:", target_crs)


# 3. Перепроецируем все слои
area_bounds = area_bounds.to_crs(target_crs)
grocery = grocery.to_crs(target_crs)
landuse = landuse.to_crs(target_crs)
public_transport = public_transport.to_crs(target_crs)
accessibility_zones = accessibility_zones.to_crs(target_crs)


# 4. Проверяем CRS после перепроецирования
print("area_bounds:", area_bounds.crs)
print("grocery:", grocery.crs)
print("landuse:", landuse.crs)
print("public_transport:", public_transport.crs)
print("accessibility_zones:", accessibility_zones.crs)

area_bounds: epsg:4326
grocery: EPSG:4326
landuse: EPSG:32637
public_transport: EPSG:32637
accessibility_zones: EPSG:4326
target_crs: EPSG:4326
area_bounds: epsg:4326
grocery: EPSG:4326
landuse: EPSG:4326
public_transport: EPSG:4326
accessibility_zones: EPSG:4326


### Шаг 3. Создание базовой карты

Определите центр карты на основе границы исследуемой территории.


In [190]:
# ваш код
center = area_bounds.geometry.centroid.iloc[0]
center_lat = center.y
center_lon = center.x

print(center_lat, center_lon)

55.75362567212317 37.70490255985149


C:\Users\AcerAspire5\AppData\Local\Temp\ipykernel_8020\1279796305.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center = area_bounds.geometry.centroid.iloc[0]


Создайте объект карты


In [191]:
# ваш код
m = folium.Map(
    location= [center_lat, center_lon],
    zoom_start=12,
    tiles = 'cartodb positron',
    control_scale=True
)
m

### Шаг 4. Добавление границы территории

Создайте стиль для отображения границы.


In [192]:
# ваш код
area_style = lambda feature: {
    'fillcolor': "#7A8EA9",
    'color': "#2F4F79",
    'weight': 2,
    'fillOpacity': 0.03,
    'opacity': 0.35,
    "dashArray": "6, 5",
}

Добавьте границу на карту как отдельный слой.


In [193]:
# ваш код
folium.GeoJson(
    area_bounds,
    name = 'Граница р-на Лефортово',
    style_function = area_style
).add_to(m)

m

### Шаг 5. Добавление слоя землепользования

Посмотрите, какие типы землепользования есть в данных.


In [194]:
# ваш код
landuse['landuse'].value_counts()

landuse
residential    52
industrial     38
commercial     16
retail          2
Name: count, dtype: int64

Создайте словарь цветов для основных типов землепользования.


In [195]:
# ваш код
landuse_colors = {
    "residential": "#fdd49e",
    "commercial": "#fc8d59",
    "industrial": "#7b7676",
    'retail': "#ee3f24",
}

In [196]:
landuse_groups = {
    "residential": "residential",

    "commercial": "commercial",
    "retail": "retail",

    "industrial": "industrial",

}

Создайте функцию стилизации.


In [197]:
# ваш код
def landuse_style(feature):

    landuse_type = feature["properties"].get("landuse")

    landuse_class = landuse_groups.get(landuse_type, "other")

    color = landuse_colors.get(landuse_class, "#d9d9d9")

    return {
        "fillColor": color,
        "color": "#ffffff",
        "weight": 0.4,
        "fillOpacity": 0.65,
        "opacity": 0.6,
    }

Добавьте подсказки (`tooltip`) при необходимости.


In [198]:
# ваш код
landuse_tooltip = folium.GeoJsonTooltip(
    fields=['landuse'],
    aliases=['Тип землепользования'],
    localize=True
)

Добавьте слой на карту.


In [199]:
# ваш код
folium.GeoJson(
    landuse,
    name = 'Землепользование',
    style_function=landuse_style,
    tooltip=landuse_tooltip
).add_to(m)

m

Добавление легенды (опционально)

Чтобы карта была более читаемой, можно добавить легенду для слоя землепользования.

В библиотеке Folium нет «готового» универсального способа создания легенд для пользовательских слоёв, поэтому вам потребуется самостоятельно разобраться:

- как добавлять HTML-элементы на карту;
- как оформлять легенду через CSS;
- как связать цвета легенды с вашими типами землепользования.

Для этого изучите документацию Folium и примеры из интернета.


In [200]:
legend_html = """
<div style="
    position: fixed;
    bottom: 30px;
    left: 30px;
    width: 220px;
    background-color: white;
    border: 1px solid grey;
    z-index: 9999;
    font-size: 14px;
    padding: 10px;
    box-shadow: 2px 2px 5px rgba(0,0,0,0.3);
">
<b>Землепользование</b><br>

<i style="background:#fdd49e; width:14px; height:14px; float:left; margin-right:8px; opacity:0.7;"></i>
Жилое<br>

<i style="background:#fc8d59; width:14px; height:14px; float:left; margin-right:8px; opacity:0.7;"></i>
Общественно-деловое<br>

<i style="background:#ee3f24; width:14px; height:14px; float:left; margin-right:8px; opacity:0.7;"></i>
Торговое<br>

<i style="background:#7b7676; width:14px; height:14px; float:left; margin-right:8px; opacity:0.7;"></i>
Промышленное<br>
</div>
"""

m.get_root().html.add_child(
    folium.Element(legend_html),
    name="landuse_legend"
)

m

### Шаг 6. Добавление объектов и соответствующих им зон доступности

На этом этапе добавьте на карту объекты и связанные с ними зоны пешеходной доступности.

Для каждой категории рекомендуется создать отдельную группу слоёв (`FeatureGroup`), внутри которой будут:

- сами объекты;
- зоны доступности 5 минут;
- зоны доступности 10 минут;
- зоны доступности 15 минут.

Зоны доступности для каждой из категорий могут быть либо в одном слое либо разделены по интервалам. Выберите тот вариант, который удобнее для управления слоями и визуального анализа.


#### 6.1. Первая категория объектов

Ниже приведён пример добавления на карту одной категории объектов и соответствующих зон доступности.

##### Создание группы слоёв


In [201]:
category_group = folium.FeatureGroup(
    name="Продуктовые магазины и их зоны доступности",
    show=True
)

Параметр `show=True` означает, что слой будет отображаться сразу после загрузки карты. Подумайте, все ли категории объектов должны отображаться по умолчанию. Если слоёв много, часть из них лучше сделать скрытыми при открытии карты с помощью `show=False`.


##### Добавление объектов

Для точечных объектов удобно использовать `CircleMarker`, но при желании в документации можно изучить и другие способы отображения:

- `Marker`;
- `Icon`;
- `BeautifyIcon`;
- иконки Font Awesome.


##### Добавление зон доступности

Дальнейшие шаги будут зависеть от того, как у вас сохранены зоны доступности:

- в одном наборе данных;
- или в нескольких слоях по временным интервалам.

Главное — добавлять зоны в ту же группу, что и объекты, с помощью:

```python
.add_to(category_group)
```

Для зон доступности рекомендуется использовать:

- один цвет для одной категории объектов;
- разную насыщенность или прозрачность для интервалов 5, 10 и 15 минут;
- более прозрачную заливку для больших зон, чтобы они не перекрывали остальные данные.

Также в подсказки (`tooltip`) или всплывающие окна (`popup`) можно добавить результаты анализа доступности населения, если эти показатели уже записаны в атрибутивной таблице зон.


Посмотрим на пример, если все зоны доступности находятся в одном наборе данных:

Например, в GeoDataFrame есть поле time, в котором указаны интервалы. В этом случае можно автоматически задавать стили и подсказки через функции.


Например, можно создать словарь цветов и прозрачности для разных временных интервалов:


In [202]:
accessibility_zones.head()

,time_min,category,population_sum,outside_population,share_total_population,outside_share_total_population,geometry
0,5,grocery,64960.0,38145.0,63.00,37.00,"POLYGON ((37.70141 55.74902, 37.70004 55.74878..."
1,10,grocery,99675.0,3430.0,96.67,3.33,"POLYGON ((37.69207 55.74838, 37.69187 55.74825..."
2,15,grocery,99675.0,3430.0,96.67,3.33,"POLYGON ((37.69207 55.74838, 37.69187 55.74825..."


In [203]:
iso_styles = {
    5: {
        "fillColor": "#e34a33",
        "color": "#e34a33",
        "fillOpacity": 0.55
    },
    10: {
        "fillColor": "#fdbb84",
        "color": "#fdbb84",
        "fillOpacity": 0.35
    },
    15: {
        "fillColor": "#fee8c8",
        "color": "#fee8c8",
        "fillOpacity": 0.15
    }
}

Функция будет автоматически выбирать стиль в зависимости от значения поля `time`.


In [204]:
def style_function(feature):

    time_value = feature["properties"]["time_min"]

    style = iso_styles[time_value]

    return {
        "fillColor": style["fillColor"],
        "color": style["color"],
        "weight": 1,
        "fillOpacity": style["fillOpacity"]
    }


Если в данных есть показатели населения, их можно выводить в подсказках.

Например:


In [205]:
tooltip = folium.GeoJsonTooltip(
    fields=[
        "time_min",
        "population_sum",
        "share_total_population"
    ],
    aliases=[
        "Время:",
        "Население в зоне:",
        "Доля населения:"
    ],
    localize=True
)

Добавление слоя на карту


In [206]:
folium.GeoJson(
    accessibility_zones,
    name="Зоны доступности продуктовых магазинов",
    style_function=style_function,
    tooltip=tooltip
).add_to(category_group)

In [207]:
for _, row in grocery.iterrows():

    folium.CircleMarker(
        location=[
            row.geometry.y,
            row.geometry.x
        ],
        radius=2.5,
        color="#FFFFFF",
        fill=True,
        fill_color="blue",
        fill_opacity=0.9,
        popup=row["name"],
        tooltip="Продуктовый магазин",
        weight=0.5,
    ).add_to(category_group)

##### Добавление группы на карту


In [208]:
category_group.add_to(m)

#### 6.2. Добавление остальных категорий

После того как вы настроили отображение первой категории, повторите те же действия для остальных категорий объектов. Для каждой категории рекомендуется:

- создать отдельный `FeatureGroup`;


In [209]:
#ваш_код

#### 6.3. Автоматизация процесса (опционально)

После добавления нескольких категорий вы заметите, что код начинает повторяться.

Подумайте, как можно автоматизировать процесс.

Например:

- создать словарь со стилями категорий;
- оформить добавление объектов в функцию;
- оформить добавление зон доступности в функцию;
- автоматически проходить по всем категориям через цикл.


In [210]:
#ваш_код

### Шаг 7. Добавление элементов управления


Обязательно добавьте переключатель слоёв.


In [211]:
# ваш код
folium.LayerControl(collapsed=False).add_to(m)

In [212]:
from folium.plugins import MousePosition

MousePosition().add_to(m)

In [213]:
from folium.plugins import Fullscreen

Fullscreen(
    position="bottomright",
    title="Открыть на весь экран",
    title_cancel="Выйти из полноэкранного режима",
    force_separate_button=True,
).add_to(m)

In [214]:
from folium.plugins import MiniMap


MiniMap(tile_layer="cartodbpositron", toggle_display=True).add_to(m)

Вы также можете добавить другие элементы по желанию:

- мини-карту;
- полноэкранный режим;
- измерение расстояний;
- легенду;
- поиск по объектам.


### Шаг 8. Просмотр, сохранение и публикация карты

Отобразите итоговую карту.


In [215]:
# ваш код
m

Сохраните карту в HTML-файл.


In [216]:
# ваш код
m.save("../index.html")

Откройте файл в браузере и проверьте, что:

- карта загружается;
- слои можно включать и выключать;
- подсказки отображаются корректно;
- карта читается визуально.


После этого вы можете опубликовать карту на GitHub Pages или другом ресурсе.


## Результат

В результате у вас должна получиться интерактивная веб-карта, на которой отображены данные, подготовленные на предыдущих этапах проекта:

- граница исследуемой территории;
- землепользование;
- объекты повседневных функций;
- остановки общественного транспорта;
- зоны пешеходной доступности;
- элементы управления картой;
- сохранённый HTML-файл с картой.
